# Spike type prediction from recording

This notebook calculates an average spike triggered wavelet spectrogram for each spike type.

## Initialization - don't change

In [ ]:
import sys
sys.path.append('..')

from workflows.run_util.io import collect_paths_from_table
from param_classes.workflows.wavelet_analysis import (
    SpikeTriggeredWaveletWorkflowParams
)
from workflows.triggered_wavelet_analysis import run_analysis
from workflows.spike_type_prediction import collect_predictive_features

params = SpikeTriggeredWaveletWorkflowParams()

## Color settings

Set the colors that will be used for coloring the markers on the strip plot and the cellwise average firing frequency curve.

In [ ]:
params.palette = {
    'cell_type': {
        'PYR_L2-3': '#803300ff',
        'PYR_L4-6': '#502d16ff',
        'RSI_L2-3': '#ff48a3ff',
        'RSI_L4-6': '#800080ff',
        'FSI_L2-3': '#d54500ff',
        'FSI_L4-6': '#7a0000ff'
    },
    'spike_type': {
        'AIS-AP': "#3150B7",
        'RA-AP': '#00ff00',
        'UNKNOWN': '#000000b2'
    }
}

## Data settings

Set raw input dir. You need to give the base path where the group directories can be found.

In [ ]:
params.data.raw_data.path='C:\\Datasets\\raap\\invivo'
params.data.raw_data.extension='mat'

Set the base path of the directory where csv files containing the spike features and categories can be found.
> It needs to be the same structure as the `raw_data` directory.

In [ ]:
params.data.spike_features.path = 'C:\\tmp\\Bozso2025\\PutRAAP\\Results\\New\\SpikeFeatures'
params.data.spike_features.extension = 'csv'

## Input-output settings

Set the Excel table containing the cell attributes, grouping and selector information.

In [ ]:
params.input = {
    'mode': 'table',
    'table_path': 'C:\\tmp\\Bozso2025\\PutRAAP\\Results\\New\\data_invivo.xlsx',
    'select_rules': [
        {'selected': 1}
    ]
}

Set the output base directory path. All results of this analysis (cellwise aggregated tables, grouped results, figures, statistics) is going to saved there.

In [ ]:
params.output.path = 'C:\\tmp\\Bozso2025\\PutRAAP\\Results\\New\\Preprocessed\\SpikeTypePrediction'

## Analysis parameters

 - `params.analysis.spike_types`: the list of the spike types to use as trigger

In [ ]:
params.analysis.spike_types = ['AIS-AP', 'UNKNOWN', 'RA-AP']

In [ ]:
type_to_id = {
    'AIS-AP': 0,
    'UNKNOWN': 1,
    'RA-AP': 2
}

## Analysis

In [ ]:
input_files = collect_paths_from_table(params.data, params.input)

In [ ]:
pool_features = collect_predictive_features(
    input_files,
    type_to_id,
    ['delta', 'spdl', 'sw', 'theta'],
    params.output.path
)

In [ ]:
pool_features

In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, f1_score

X, y = pool_features.iloc[:, 3:], pool_features['y']
nan_flag = X.isna().any(axis=1)
X, y = X[~nan_flag], y[~nan_flag]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

gnb = GaussianNB()
y_pred = gnb.fit(X_train, y_train).predict(X_test)
cfm = confusion_matrix(y_test, y_pred)
print(cfm)
print(f1_score(y_test, y_pred, average='weighted'))